# Router: fine-tune Qwen3-4B (LoRA) để nhận ra dạng của một đề bài (chỉ huấn luyện)

Bản sao notebook huấn luyện Qwen3-0.6B (bản đạt 100%), giữ nguyên dữ liệu và mọi thiết lập huấn luyện. Chỉ khác:

1. `MODEL = Qwen/Qwen3-4B`.
2. Bỏ Dạng 122 (Số phức D34) khỏi dữ liệu huấn luyện và danh sách dạng: còn 141 dạng, 12690 cặp.
3. Trên T4 nạp trọng số float16 thay cho float32 (float32 của 4B chiếm khoảng 16 GB, bằng cả bộ nhớ T4).
4. `MAX_TRAIN_MINUTES = 680` (bản cũ 270): dùng gần hết 12 giờ của một phiên Kaggle.
5. Chỉ huấn luyện và lưu adapter; phần đánh giá chạy ở notebook riêng.
6. Dữ liệu huấn luyện tự tải từ GitHub (commit cố định, có kiểm tra mã băm): không cần gắn Dataset.

## Cách chạy

1. **Settings**: Accelerator = **GPU T4 x2** (notebook dùng 1 GPU, như lần chạy 0.6B); **Internet = On**.
2. Chạy thử: đặt `QUICK_TEST = True` ở ô cấu hình (đổi `'0'` thành `'1'` ở dòng `QUICK_TEST`), **Run All**. Dòng
   `step … s/step` cho tốc độ; chạy thử dùng 8 câu mỗi bước, chạy thật 32 câu mỗi bước.
3. Chạy thật: `QUICK_TEST = False`, **Save Version → Save & Run All (Commit)**.
4. Kết quả trong **Output**: `router_lora/` (adapter), `train_progress.log`, `router_ckpt/` (checkpoint).
   Nếu log có dòng `time limit reached`, huấn luyện chưa đủ số bước và adapter chưa phải bản cuối.


In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate bitsandbytes
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'                   # ONE GPU. With two, Trainer doubles the batch (16 x 2 GPUs = 32 per step, 64 per update)
import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__,
      '| GPUs:', torch.cuda.device_count())


In [ ]:
import os, re, json, glob, copy, time, random
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')    # less memory fragmentation (must be set before torch is imported)
from collections import Counter
import numpy as np
import torch

# ------------------------------------------------------------------ configuration
MODEL = os.environ.get('ROUTER_MODEL', 'Qwen/Qwen3-4B')
QUICK_TEST = os.environ.get('ROUTER_QUICK', '0') == '1'    # True: a few steps and a few questions, only to check that everything runs

LOSS_ON = os.environ.get('ROUTER_LOSS', 'completion')      # 'completion' (recommended) or 'all'
USE_4BIT = False                                          # QLoRA (4-bit base model). False = LoRA on the 16-bit model
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EPOCHS = 2
MAX_TRAIN_MINUTES = 680                                    # Kaggle stops a session at 12 h (720 min); the rest is for setup, the final validation and saving
SCORE_MAX_MINUTES = 200                                   # safety: scoring stops after this many minutes (the questions are scored in random order)
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH, GRAD_ACCUM = int(os.environ.get('ROUTER_BS', '16')), int(os.environ.get('ROUTER_ACCUM', '2'))                      # effective batch: 32 sequences (the loss no longer builds the full logits, see RouterTrainer)
MAX_LEN = 512                                             # pairs longer than this are left out of training (1 of 12780: 726 tokens); it keeps the memory of a batch bounded
VAL_PER_TYPE = 3                                          # pairs of every type kept out of training to watch the loss (0 = none)
SEED = 42

# Evaluation (main): for every test question the probability of each of the 142 answers 'Dạng N: mô tả' is computed and the
# most probable one is the model's answer. Free generation is kept only as a comparison on a subset.
SCORE_CANDIDATES = os.environ.get('ROUTER_SCORE', '1') == '1'
SCORE_ALL = os.environ.get('ROUTER_SCORE_ALL', '1') == '1'                                          # score the whole test set (12951 questions: 172 source + 12779 variants)
SCORE_PER_TYPE = int(os.environ.get('ROUTER_SCORE_PER_TYPE', '20'))                                       # variants of every type in the sample (20 x 142 = 2840; all source questions are always included)
GREEDY_COMPARE = os.environ.get('ROUTER_GREEDY', '1') == '1'                                     # also let the model write its answer freely on that subset and read 'Dạng N' from it

assert LOSS_ON in ('completion', 'all')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def find(name):
    d = os.environ.get('ROUTER_DATA_DIR')
    if d:
        p = os.path.join(d, name)
        assert os.path.exists(p), p
        return p
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input: attach the dataset to the notebook first.'
    return hits[0]

# the training data is downloaded from the repo at a fixed commit (no Kaggle Dataset needed; Internet must be On)
import hashlib, urllib.request
RAW = 'https://raw.githubusercontent.com/manhdungcr7/slm-highschool-math-tutoring/b213dc43036660918fc860f5df02b61ac30a9a16/data/router_train/kaggle_bundle_v3/'
DATA_DIR = '/tmp/router_data' if os.path.isdir('/kaggle/working') else os.environ.get('ROUTER_DATA_TMP', './router_data')
os.makedirs(DATA_DIR, exist_ok=True)
def fetch(name, sha):
    path = os.path.join(DATA_DIR, name)
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(RAW + name, path); break
        except Exception as e:
            print('retry', name, repr(e)[:80]); time.sleep(5)
    data = open(path, 'rb').read()
    assert hashlib.sha1(b'blob %d\0' % len(data) + data).hexdigest() == sha, f'{name} is not the expected version'
    return path
TRAIN_PATH = fetch('finetune_router.jsonl', '80e65d755380c47595906479b9ef4d21b10b220c')
CAND_PATH = fetch('candidates.json', '75b57b65e9fa19dc6e6d9895fab15bb8c49f9308')
EVAL_PATH = None                                           # training only: the evaluation runs in a separate notebook
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.environ.get('ROUTER_OUT', './router_out')
os.makedirs(OUT_DIR, exist_ok=True)
LORA_DIR = os.path.join(OUT_DIR, 'router_lora')
print('model:', MODEL, '| loss on:', LOSS_ON, '| quick test:', QUICK_TEST)
print('data :', TRAIN_PATH, EVAL_PATH, CAND_PATH, sep='\n       ')


In [ ]:
# ------------------------------------------------------------------ data
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = 'right'
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
EOS = '<|im_end|>'

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_all = read_jsonl(TRAIN_PATH)
CANDS = json.load(open(CAND_PATH, encoding='utf-8'))
assert len(CANDS) == 142 and all(c.startswith(f'Dạng {i + 1}:') for i, c in enumerate(CANDS))
assert all(r['completion'] in set(CANDS) for r in train_all), 'a training answer is not one of the 142 candidates'
# Dạng 122 (Số phức D34) is removed from the thesis: left out of the training pairs and of the candidates
REMOVED = {122}
CANDS = [c for c in CANDS if int(re.match(r'Dạng (\d+):', c).group(1)) not in REMOVED]
CAND_NUM = [int(re.match(r'Dạng (\d+):', c).group(1)) for c in CANDS]      # candidate index -> type number (122 is skipped)
train_all = [r for r in train_all if r['completion'] in set(CANDS)]
assert len(CANDS) == 141 and len(train_all) == 12690

def type_of(completion):
    return int(re.match(r'Dạng (\d+):', completion).group(1))

# keep VAL_PER_TYPE pairs of every type out of the training set (only to watch the loss)
by_type = {}
for r in train_all:
    by_type.setdefault(type_of(r['completion']), []).append(r)
rng = random.Random(SEED)
train_rows, val_rows = [], []
for n, rs in sorted(by_type.items()):
    rs = rs[:]; rng.shuffle(rs)
    val_rows += rs[:VAL_PER_TYPE]
    train_rows += rs[VAL_PER_TYPE:]
rng.shuffle(train_rows)
if QUICK_TEST:
    train_rows, val_rows = train_rows[:160], val_rows[:40]
print('training pairs:', len(train_rows), '| validation pairs:', len(val_rows), '| types:', len(by_type))

def prompt_text(user):
    # chat template with the thinking block closed and empty: the answer starts right after it
    return tok.apply_chat_template([{'role': 'user', 'content': user}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)

def encode_pair(user, completion):
    p = tok(prompt_text(user), add_special_tokens=False).input_ids
    c = tok(completion + EOS, add_special_tokens=False).input_ids
    ids = p + c
    labels = ([-100] * len(p) + c) if LOSS_ON == 'completion' else list(ids)
    return {'input_ids': ids[:MAX_LEN], 'labels': labels[:MAX_LEN]}

train_ds = [encode_pair(r['prompt'], r['completion']) for r in train_rows]
n_before = len(train_ds)
train_ds = [x for x in train_ds if len(x['input_ids']) < MAX_LEN]            # a cut pair would lose the answer, so it is dropped instead
print(f'pairs left out because they are longer than {MAX_LEN} tokens: {n_before - len(train_ds)}')
val_ds = [encode_pair(r['prompt'], r['completion']) for r in val_rows]

lens = np.array([len(x['input_ids']) for x in train_ds])
n_loss = sum(sum(1 for t in x['labels'] if t != -100) for x in train_ds)
print(f'sequence length: mean {lens.mean():.0f}, p99 {np.percentile(lens, 99):.0f}, max {lens.max()} (MAX_LEN={MAX_LEN})')
print(f'tokens per epoch: {lens.sum():,} | tokens that carry loss: {n_loss:,} ({n_loss / lens.sum():.1%})')
assert lens.max() <= MAX_LEN
# look at one example: the part the loss is computed on
ex = train_ds[0]
print('--- loss is computed on ---')
print(tok.decode([t for t in ex['labels'] if t != -100]))

class DS(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    m = max(len(b['input_ids']) for b in batch)
    ids = torch.full((len(batch), m), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b['input_ids'])
        ids[i, :n] = torch.tensor(b['input_ids']); lab[i, :n] = torch.tensor(b['labels']); att[i, :n] = 1
    return {'input_ids': ids, 'labels': lab, 'attention_mask': att}


In [ ]:
# ------------------------------------------------------------------ model: 4-bit base + LoRA adapters
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

HAS_CUDA = torch.cuda.is_available()
# A T4 (capability 7.5) has no bfloat16 tensor cores, yet torch.cuda.is_bf16_supported() answers True because it emulates them,
# which is several times slower. So bfloat16 is used only from capability 8.0 (Ampere) on; otherwise float16.
BF16 = HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8 and os.environ.get('ROUTER_FP16', '0') != '1'
COMPUTE_DTYPE = torch.bfloat16 if BF16 else (torch.float16 if HAS_CUDA else torch.float32)
# float16 training: the weights stay in float32 (safe against overflow) and the matrix products run in float16 (autocast)
LOAD_DTYPE = COMPUTE_DTYPE                                 # Qwen3-4B: float32 weights (16 GB) do not fit a T4, so float16
print('GPUs visible:', torch.cuda.device_count(), '| bfloat16:', BF16, '| weights loaded as:', LOAD_DTYPE, '| matrix products in:', COMPUTE_DTYPE)
if USE_4BIT and not HAS_CUDA:
    USE_4BIT = False                                      # 4-bit needs a GPU
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=COMPUTE_DTYPE) if USE_4BIT else None
kw = dict(quantization_config=quant, device_map={'': 0} if HAS_CUDA else None)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=LOAD_DTYPE, **kw)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=LOAD_DTYPE, **kw)
model.config.use_cache = False
if USE_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={'use_reentrant': False})
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()
if HAS_CUDA:
    print(f'GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


In [ ]:
# ------------------------------------------------------------------ training
from transformers import Trainer, TrainingArguments

steps_per_epoch = max(1, len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM))
total_steps = int(os.environ.get('ROUTER_QUICK_STEPS', '12')) if QUICK_TEST else steps_per_epoch * EPOCHS
print(f'steps per epoch: {steps_per_epoch} | total optimizer steps: {total_steps}')

def make_args(**extra):
    base = dict(
        output_dir=os.path.join(OUT_DIR, 'router_ckpt'),
        per_device_train_batch_size=PER_DEVICE_BATCH if not QUICK_TEST else 4,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM if not QUICK_TEST else 2,
        max_steps=total_steps,
        learning_rate=LEARNING_RATE, lr_scheduler_type='cosine', weight_decay=0.0,
        logging_steps=5 if QUICK_TEST else 20,
        save_strategy='no' if QUICK_TEST else 'steps', save_steps=400, save_total_limit=2,
        bf16=BF16, fp16=(HAS_CUDA and not BF16),
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch',
        report_to='none', prediction_loss_only=True, seed=SEED, remove_unused_columns=False, dataloader_num_workers=0 if (QUICK_TEST or not HAS_CUDA) else int(os.environ.get('ROUTER_WORKERS', '2')),
    )
    base.update(extra)
    return base

eval_kw = dict(eval_strategy='steps' if len(val_ds) else 'no', eval_steps=5 if QUICK_TEST else 200)
args = TrainingArguments(**make_args(**eval_kw, warmup_steps=max(1, int(0.03 * total_steps))))

from transformers import TrainerCallback

class Progress(TrainerCallback):
    """Prints the speed and the time left every 10 steps, writes the same to a file, and stops training after MAX_TRAIN_MINUTES."""
    def __init__(self):
        self.t0 = time.time()
        self.path = os.path.join(OUT_DIR, 'train_progress.log')
        self.time_up = False
        self.stop_at = None                                  # end of the current chunk: stop and save a checkpoint there
    def on_step_end(self, args, state, control, **kw):
        el = time.time() - self.t0
        if state.global_step % 10 == 0 or state.global_step == 1:
            rate = state.global_step / max(el, 1e-9)
            msg = f'step {state.global_step}/{state.max_steps} | {el / 60:.1f} min | {1 / max(rate, 1e-9):.1f} s/step | about {(state.max_steps - state.global_step) / max(rate, 1e-9) / 60:.0f} min left'
            print(msg, flush=True)
            open(self.path, 'a').write(msg + chr(10))
        if self.stop_at is not None and state.global_step >= self.stop_at:
            control.should_training_stop = True
            control.should_save = True
        if el > MAX_TRAIN_MINUTES * 60:
            print('time limit reached: training stops here', flush=True)
            self.time_up = True
            control.should_training_stop = True
            control.should_save = True

class RouterTrainer(Trainer):
    """Loss on the answer tokens only, WITHOUT building the logits of every position.
    The vocabulary has 151936 tokens, so the logits of a whole batch in float32 take several GB and overflowed a 16 GB T4
    (the answer is only about a fifth of each sequence). Here the last hidden states are taken, only the positions that
    predict an answer token go through the output layer, and the cross entropy is computed on those."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if LOSS_ON != 'completion':
            return super().compute_loss(model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch)
        causal = self.accelerator.unwrap_model(model)
        causal = causal.get_base_model() if hasattr(causal, 'get_base_model') else causal      # Qwen3ForCausalLM inside the PEFT wrapper
        hidden = causal.model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], use_cache=False).last_hidden_state
        target = inputs['labels'][:, 1:]
        take = target != -100
        logits = causal.lm_head(hidden[:, :-1][take]).float()                                  # only the answer positions
        loss = torch.nn.functional.cross_entropy(logits, target[take], reduction='sum')
        loss = loss / (num_items_in_batch if num_items_in_batch is not None else take.sum().clamp(min=1))
        return (loss, {'logits': logits}) if return_outputs else loss

PROG = Progress()
trainer = RouterTrainer(model=model, args=args, train_dataset=DS(train_ds),
                  eval_dataset=DS(val_ds) if len(val_ds) else None, data_collator=collate, callbacks=[PROG])
N_CHUNKS = 4                                             # the training is run by the next cells, one chunk each, so that every cell ends (and shows its log) in turn
TRAIN_T0 = time.time()


In [ ]:
# ------------------------------------------------------------------ training, part 1 of 4
if trainer.state.global_step < total_steps and not PROG.time_up:
    PROG.stop_at = min(total_steps, -(-total_steps * 1 // N_CHUNKS))
    trainer.train(resume_from_checkpoint=None)
    print(f'part 1 of 4 done at step ' + str(trainer.state.global_step) + f' | {(time.time() - TRAIN_T0) / 60:.1f} min since the start of training')
else:
    print('nothing left to train')


In [ ]:
# ------------------------------------------------------------------ training, part 2 of 4
if trainer.state.global_step < total_steps and not PROG.time_up:
    PROG.stop_at = min(total_steps, -(-total_steps * 2 // N_CHUNKS))
    trainer.train(resume_from_checkpoint=True)
    print(f'part 2 of 4 done at step ' + str(trainer.state.global_step) + f' | {(time.time() - TRAIN_T0) / 60:.1f} min since the start of training')
else:
    print('nothing left to train')


In [ ]:
# ------------------------------------------------------------------ training, part 3 of 4
if trainer.state.global_step < total_steps and not PROG.time_up:
    PROG.stop_at = min(total_steps, -(-total_steps * 3 // N_CHUNKS))
    trainer.train(resume_from_checkpoint=True)
    print(f'part 3 of 4 done at step ' + str(trainer.state.global_step) + f' | {(time.time() - TRAIN_T0) / 60:.1f} min since the start of training')
else:
    print('nothing left to train')


In [ ]:
# ------------------------------------------------------------------ training, part 4 of 4
if trainer.state.global_step < total_steps and not PROG.time_up:
    PROG.stop_at = min(total_steps, -(-total_steps * 4 // N_CHUNKS))
    trainer.train(resume_from_checkpoint=True)
    print(f'part 4 of 4 done at step ' + str(trainer.state.global_step) + f' | {(time.time() - TRAIN_T0) / 60:.1f} min since the start of training')
else:
    print('nothing left to train')


In [ ]:
# ------------------------------------------------------------------ end of training: validation, save the adapter
print(f'training time: {(time.time() - TRAIN_T0) / 60:.1f} min')
if len(val_ds):
    print('final validation:', trainer.evaluate())

model.save_pretrained(LORA_DIR)
tok.save_pretrained(LORA_DIR)
print('adapter saved to', LORA_DIR, os.listdir(LORA_DIR))
import gc
del trainer
gc.collect()
if HAS_CUDA:
    torch.cuda.empty_cache()
